# Analyse des ventes — Pandas & Matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

Path("figures").mkdir(exist_ok=True)
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
                     "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]),
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.bbox": "tight"})

## 1. DataFrames Settings

In [3]:
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 20)
pd.set_option("display.width", 240)

pd.DataFrame(np.random.rand(200, 20))

,0,1,2,3,4,5,...,14,15,16,17,18,19
0,0.950820,0.339561,0.170172,0.033955,0.902636,0.436527,...,0.159434,0.193437,0.478961,0.988721,0.397181,0.130851
1,0.820940,0.731694,0.575570,0.003807,0.569192,0.064053,...,0.876101,0.817313,0.650864,0.288374,0.684230,0.127929
2,0.272063,0.899399,0.093057,0.512555,0.610044,0.114011,...,0.250106,0.161658,0.367736,0.910872,0.500680,0.925972
3,0.413553,0.323179,0.332336,0.613384,0.348612,0.283976,...,0.438444,0.499365,0.356532,0.401405,0.047285,0.589338
4,0.968058,0.105175,0.161693,0.786841,0.993339,0.841946,...,0.091978,0.468749,0.431089,0.360157,0.437868,0.434846
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.666558,0.513743,0.001353,0.095403,0.196620,0.392034,...,0.429211,0.551906,0.561957,0.584822,0.235343,0.774352
196,0.808792,0.344972,0.643166,0.908874,0.205554,0.702916,...,0.255457,0.893863,0.056163,0.963318,0.428465,0.407978
197,0.613401,0.401165,0.727337,0.931884,0.047320,0.981625,...,0.040565,0.131563,0.556799,0.547612,0.446792,0.828605
198,0.557504,0.777099,0.384441,0.548963,0.322738,0.840301,...,0.513201,0.990856,0.075133,0.957836,0.999900,0.665075


- **Series** : une colonne de valeurs d'un seul type, avec des étiquettes.
- **DataFrame** : un tableau = plusieurs Series qui partagent le même index.
- **Index** : Axe d'étiquette, identifie une ligne.

## 2. Chargement et exploration

In [6]:
xls = pd.read_excel("data/CC-AIBD-DataAnalyse-donnees.xlsx", sheet_name=None)
orders      = xls["Sales Order_data"]
territories = xls["Sales Territory_data"]
sales       = xls["Sales_data"]
resellers   = xls["Reseller_data"]
dates       = xls["Date_data"]
products    = xls["Product_data"]
customers   = xls["Customer_data"]
tables = {"orders": orders, "territories": territories, "sales": sales, "resellers": resellers,
          "dates": dates, "products": products, "customers": customers}

for name, df in tables.items():
    print(f"\n===== {name} =====")
    display(df.head())
    df.info()
    display(df.describe())


===== orders =====


,Channel,SalesOrderLineKey,Sales Order,Sales Order Line
0,Reseller,43659001,SO43659,SO43659 - 1
1,Reseller,43659002,SO43659,SO43659 - 2
2,Reseller,43659003,SO43659,SO43659 - 3
3,Reseller,43659004,SO43659,SO43659 - 4
4,Reseller,43659005,SO43659,SO43659 - 5


<class 'pandas.DataFrame'>
RangeIndex: 121253 entries, 0 to 121252
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Channel            121253 non-null  str  
 1   SalesOrderLineKey  121253 non-null  int64
 2   Sales Order        121253 non-null  str  
 3   Sales Order Line   121253 non-null  str  
dtypes: int64(1), str(3)
memory usage: 3.7 MB


,SalesOrderLineKey
count,1.212530e+05
mean,5.782642e+07
std,9.009990e+06
min,4.365900e+07
25%,4.987905e+07
50%,5.702800e+07
75%,6.549000e+07
max,7.512300e+07



===== territories =====


...

,ProductKey,Standard Cost,List Price
count,397.000000,397.000000,397.000000
mean,408.000000,436.454083,751.106791
std,114.748275,497.076884,837.991094
min,210.000000,0.856500,2.290000
25%,309.000000,37.120900,69.990000
50%,408.000000,204.625100,364.090000
75%,507.000000,660.914200,1214.850000
max,606.000000,2171.294200,3578.270000



===== customers =====


,CustomerKey,Customer ID,Customer,City,State-Province,Country-Region,Postal Code
0,-1,[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable]
1,11000,AW00011000,Jon Yang,Rockhampton,Queensland,Australia,4700
2,11001,AW00011001,Eugene Huang,Seaford,Victoria,Australia,3198
3,11002,AW00011002,Ruben Torres,Hobart,Tasmania,Australia,7001
4,11003,AW00011003,Christy Zhu,North Ryde,New South Wales,Australia,2113


<class 'pandas.DataFrame'>
RangeIndex: 18485 entries, 0 to 18484
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   CustomerKey     18485 non-null  int64
 1   Customer ID     18485 non-null  str  
 2   Customer        18485 non-null  str  
 3   City            18485 non-null  str  
 4   State-Province  18485 non-null  str  
 5   Country-Region  18485 non-null  str  
 6   Postal Code     18485 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1011.0 KB


,CustomerKey
count,18485.000000
mean,20240.404923
std,5337.947953
min,-1.000000
25%,15620.000000
50%,20241.000000
75%,24862.000000
max,29483.000000


Types à corriger, tous dans `sales` (page `Sales_data` du fichier excel) :
- `OrderDateKey` et `DueDateKey` : dates stockées en entier `AAAAMMJJ` → à convertir en `datetime`.
- `ShipDateKey` : dates stockées en entier `AAAAMMJJ` mais en `float` → à convertir en `datetime`.
- `Unit Price Discount Pct` : entier alors que c'est un pourcentage → à convertir en decimal.
- `ResellerKey`, `CustomerKey` : la valeur `-1` signifie « aucun » (voir la ligne `[Not Applicable]` dans `resellers` et `customers`) → à exclure avant d'analyser les clients ou les revendeurs.

In [8]:
cost = products["Standard Cost"] #On extrait la colonne du cout de fabrication de chaque produit
print("Médiane :", cost.median(), "| Écart-type :", cost.std().__round__(2), "| Variance :", cost.var().__round__(2))
products.groupby("Category")["Standard Cost"].agg(["median", "std", "var"])

Médiane : 204.6251 | Écart-type : 497.08 | Variance : 247085.43


,median,std,var
Category,,,
Accessories,11.2163,13.127986,172.344014
Bikes,713.0798,552.407211,305153.726769
Clothing,26.1763,12.424381,154.365235
Components,187.1571,280.636563,78756.880441
